# ABBA4 configuration comparison in the real PHI_2 potential

This experiment compares the complete selected set of sixteen fourth-order
implicit ABBA configurations in the real GC2D `PHI_2.h5` potential. The
configuration axes are:

- two fourth-order projection placements: three projected composition
  substeps and one projection after the complete composition;
- two duplicated state spaces: the shared-time splitting in
  $\mathbb{R}^{6}$ and the fully extended splitting in
  $\mathbb{R}^{8}$;
- two projection formulations: reduced multiplier and simultaneous state
  plus multiplier; and
- two nonlinear solvers: Newton and Broyden.

Thus the experiment contains $2\times2\times2\times2=16$
configurations. Each configuration advances the same three initial
conditions independently because the fully extended implementation is a
one-particle map. This produces $16\times3=48$ independently integrated
trajectories.

### Why $\mathbb{R}^{12}$ appears

For one fully extended particle, the accepted state is
$Z=(x,y,t,k)\in\mathbb{R}^{4}$ and its duplicated splitting variables are
$(U,V)\in\mathbb{R}^{8}$. In the simultaneous projection formulation, the
nonlinear solver determines the final copies and the multiplier together:

$$
(U_f,V_f,\mu)\in
\mathbb{R}^{4}\times\mathbb{R}^{4}\times\mathbb{R}^{4}
=\mathbb{R}^{12}.
$$

Therefore $\mathbb{R}^{12}$ is only the algebraic unknown space inside
Newton or Broyden. It is not the evolved state space: the accepted state
remains in $\mathbb{R}^{4}$ and the duplicated fully extended map remains in
$\mathbb{R}^{8}$. The reduced-multiplier formulation solves only for
$\mu\in\mathbb{R}^{4}$.

The physical trajectory error uses the Euclidean convention of the stored
DOP853 reference and its independent Radau audit. The HDF5 field is clipped,
not periodically wrapped, so a minimum-image distance would be incorrect.

In [ ]:
from pathlib import Path
import hashlib

import h5py
import numpy as np

from diagnostics import load_reference_trajectory
from diagnostics.paths import find_project_root
from potential import GC2DH5Potential, load_gc2d_h5_potential
from studies import (
    ABBA4ConfigurationComparisonConfig,
    HighPrecisionReferenceConfig,
    latin_hypercube_gc_configuration,
    run_abba4_configuration_comparison,
    run_high_precision_reference_trajectory,
)
from visualization import (
    animate_abba4_configuration_trajectories,
    display_animation,
    display_records_table,
)

## Reproducible configuration

`FULL_STUDY=True` selects the scientific experiment on $[0,4]$. For a
short execution check, change only that flag to `False`; the quick profile
uses $[0,0.05]$ while retaining the real HDF5 potential, the same three
initial conditions, the same integration step and tolerances, and the
matching prefix of the certified reference.

The ignored reference directory is reused when present. If it is absent, the
notebook regenerates it reproducibly with the original DOP853 solve and an
independent Radau audit over the complete interval $[0,4]$. This bootstrap can
take several minutes even in quick mode, but it happens only once. Potential
loading, reference loading or generation, and animation rendering are outside
every per-configuration runtime. Each reported runtime is the sum of the three
independent integrations for that configuration.

In [ ]:
FULL_STUDY = True

NOTEBOOK_PATH = Path(
    "notebooks/experiments/implicit_abba/"
    "compare_abba4_r6_r8_configurations.ipynb"
)
PROJECT_ROOT = find_project_root(Path.cwd())
POTENTIAL_RELATIVE_PATH = Path("data/potential/V1/PHI_2.h5")
POTENTIAL_PATH = PROJECT_ROOT / POTENTIAL_RELATIVE_PATH
REFERENCE_SOURCE_NOTEBOOK_PATH = NOTEBOOK_PATH
REFERENCE_NAME = "phi_2_heterogeneous_3"
REFERENCE_VERSION = "v1"
REFERENCE_DIRECTORY = (
    PROJECT_ROOT
    / "outputs/developements/accuracy/phi_2_heterogeneous_3/v1"
)

B = 1.5
FIELD_INDICES = (0, 1)
GRID_NX = 128
GRID_NY = 128
DENOISING = False
DENOISING_SIGMA = 1.0
INTERPOLATION_ORDER = 3

PARTICLE_COUNT = 3
INITIAL_CONDITION_SEED = 20260827
DOMAIN_MARGIN_FRACTION = 0.05
RHO = 0.0

REFERENCE_T_SPAN = (0.0, 4.0)
REFERENCE_SAVE_INTERVAL = 0.01
DOP853_RELATIVE_TOLERANCE = 1e-12
DOP853_ABSOLUTE_TOLERANCE = 1e-14
DOP853_STEPS_PER_FIELD_PERIOD = 8
RADAU_RELATIVE_TOLERANCE = 1e-12
RADAU_ABSOLUTE_TOLERANCE = 1e-14
RADAU_STEPS_PER_FIELD_PERIOD = 16

T_SPAN = (0.0, 4.0 if FULL_STUDY else 0.05)
INTEGRATION_STEP = 0.0025
SAVE_INTERVAL = 0.01
ABSOLUTE_TOLERANCE = 1e-14
RELATIVE_TOLERANCE = 1e-13
MAX_ITERATIONS = 40
PROGRESS = False

print(f"Notebook: {NOTEBOOK_PATH}")
print(f"Study mode: {'full' if FULL_STUDY else 'quick validation'}")

## Real potential, certified reference, and initial conditions

The source file is a Git LFS asset. The HDF5 check below distinguishes the
real 600 MB field from an unresolved LFS pointer. Its checksum, selected
mode, interpolation grid, and deterministic Latin-hypercube initial state
define the complete potential and initial-condition metadata.

If the ignored reference artifact is missing, the public reference-study API
recreates the same `phi_2_heterogeneous_3/v1` directory. DOP853 uses eight
maximum steps per field period and Radau uses sixteen, both with relative
tolerance $10^{-12}$ and absolute tolerance $10^{-14}$. The regenerated
artifact retains the Euclidean distance convention required by this
non-periodic HDF5 field.

In [ ]:
if not POTENTIAL_PATH.is_file() or not h5py.is_hdf5(POTENTIAL_PATH):
    raise FileNotFoundError(
        "The real PHI_2 HDF5 asset is unavailable. Fetch the Git LFS file at "
        f"{POTENTIAL_PATH}."
    )

with POTENTIAL_PATH.open("rb") as stream:
    potential_source_sha256 = hashlib.file_digest(stream, "sha256").hexdigest()

potential = load_gc2d_h5_potential(
    POTENTIAL_PATH,
    B=B,
    indx=FIELD_INDICES,
    nx=GRID_NX,
    ny=GRID_NY,
    denoising=DENOISING,
    sigma=DENOISING_SIGMA,
    interpolation_order=INTERPOLATION_ORDER,
)
initial_configuration = latin_hypercube_gc_configuration(
    potential,
    particle_count=PARTICLE_COUNT,
    seed=INITIAL_CONDITION_SEED,
    domain_margin_fraction=DOMAIN_MARGIN_FRACTION,
)

grid = potential.grid
domain_min = np.asarray((grid.xmin, grid.ymin))
domain_max = np.asarray((grid.xmax, grid.ymax))
domain_span = domain_max - domain_min
potential_metadata = {
    "format": "GC2D_HDF5",
    "source_path": str(POTENTIAL_RELATIVE_PATH),
    "source_sha256": potential_source_sha256,
    "B": B,
    "field_indices": FIELD_INDICES,
    "nx": GRID_NX,
    "ny": GRID_NY,
    "denoising": DENOISING,
    "denoising_sigma": DENOISING_SIGMA,
    "interpolation_order": INTERPOLATION_ORDER,
    "selected_source_field_indices": potential.source_field_indices,
    "selected_frequencies": potential.frequencies,
    "normalization_factor": potential.normalization_factor,
}
initial_condition_metadata = {
    "particle_count": PARTICLE_COUNT,
    "seed": INITIAL_CONDITION_SEED,
    "domain_margin_fraction": DOMAIN_MARGIN_FRACTION,
    "sampling_min": domain_min + DOMAIN_MARGIN_FRACTION * domain_span,
    "sampling_max": domain_max - DOMAIN_MARGIN_FRACTION * domain_span,
    "sampling": "seeded_latin_hypercube_independent_axis_jitter",
}

field_frequency = float(np.min(np.abs(potential.frequencies)))
field_period = 2.0 * np.pi / field_frequency
reference_config = HighPrecisionReferenceConfig(
    t_span=REFERENCE_T_SPAN,
    save_interval=REFERENCE_SAVE_INTERVAL,
    rho=RHO,
    distance_convention="euclidean",
    relative_tolerance=DOP853_RELATIVE_TOLERANCE,
    absolute_tolerance=DOP853_ABSOLUTE_TOLERANCE,
    maximum_step=field_period / DOP853_STEPS_PER_FIELD_PERIOD,
    audit_relative_tolerance=RADAU_RELATIVE_TOLERANCE,
    audit_absolute_tolerance=RADAU_ABSOLUTE_TOLERANCE,
    audit_maximum_step=field_period / RADAU_STEPS_PER_FIELD_PERIOD,
)

if REFERENCE_DIRECTORY.is_dir():
    reference = load_reference_trajectory(REFERENCE_DIRECTORY)
    reference_origin = "loaded existing artifact"
else:
    reference_result = run_high_precision_reference_trajectory(
        potential,
        initial_configuration,
        notebook_path=REFERENCE_SOURCE_NOTEBOOK_PATH,
        config=reference_config,
        potential_metadata=potential_metadata,
        initial_condition_metadata=initial_condition_metadata,
        reference_name=REFERENCE_NAME,
        version=REFERENCE_VERSION,
        project_root=PROJECT_ROOT,
        overwrite=False,
    )
    reference = reference_result.trajectory
    reference_origin = "regenerated with DOP853 and Radau"

assert isinstance(potential, GC2DH5Potential)
assert reference.paths.directory.resolve() == REFERENCE_DIRECTORY.resolve()
assert reference.metadata["potential"]["source_sha256"] == potential_source_sha256
assert reference.metadata["config"]["distance_convention"] == "euclidean"
assert tuple(reference.metadata["config"]["t_span"]) == REFERENCE_T_SPAN
assert reference.metadata["config"]["save_interval"] == REFERENCE_SAVE_INTERVAL
assert reference.metadata["config"]["relative_tolerance"] == DOP853_RELATIVE_TOLERANCE
assert reference.metadata["config"]["absolute_tolerance"] == DOP853_ABSOLUTE_TOLERANCE
assert reference.metadata["config"]["audit_relative_tolerance"] == RADAU_RELATIVE_TOLERANCE
assert reference.metadata["config"]["audit_absolute_tolerance"] == RADAU_ABSOLUTE_TOLERANCE
assert reference.metadata["reference_name"] == REFERENCE_NAME
assert reference.metadata["particle_count"] == PARTICLE_COUNT
assert np.array_equal(initial_configuration.initial_state, reference.initial_state)

print(f"Potential: {POTENTIAL_RELATIVE_PATH}")
print(f"SHA-256: {potential_source_sha256}")
print(f"Selected source fields: {potential.source_field_indices.tolist()}")
print(f"Selected frequencies: {potential.frequencies.tolist()}")
print(f"Interpolated grid: {potential.grid.shape}")
print(f"Reference: {reference.paths.directory} ({reference_origin})")
print(f"Reference DOP853 maximum step: {reference_config.maximum_step:.16e}")
print(f"Reference Radau maximum step: {reference_config.audit_maximum_step:.16e}")
print(f"Reference audit global RMS: {reference.metadata['audit']['global_rms_distance']:.9e}")

## Sixteen configurations and 48 integrations

The study constructs every configuration once, then integrates the three
one-particle problems independently. All trajectories use the same physical
initial state, output nodes, solver tolerances, and reference samples. The
comparison itself persists no diagnostics or result files; only the certified
reference bootstrap above writes its stable ignored artifact when that artifact
is absent.

In [ ]:
comparison_config = ABBA4ConfigurationComparisonConfig(
    t_span=T_SPAN,
    particle_count=PARTICLE_COUNT,
    integration_step=INTEGRATION_STEP,
    save_interval=SAVE_INTERVAL,
    rho=RHO,
    absolute_tolerance=ABSOLUTE_TOLERANCE,
    relative_tolerance=RELATIVE_TOLERANCE,
    max_iterations=MAX_ITERATIONS,
    progress=PROGRESS,
)

assert comparison_config.particle_count == PARTICLE_COUNT

print(comparison_config)
print(f"Complete steps per trajectory: {comparison_config.step_count}")
print(f"Saved samples per trajectory: {comparison_config.output_sample_count}")

In [ ]:
result = run_abba4_configuration_comparison(
    potential,
    initial_configuration,
    reference,
    config=comparison_config,
)

## Contract checks

These checks make the intended experiment shape executable: sixteen ordered
variants, three independent solutions per variant, $16\times3=48$ trajectories
in total,
one common time grid, and exactly five numerical metrics per table row.

In [ ]:
summary_rows = result.summaries()
metric_fields = (
    "mean_trajectory_error",
    "final_trajectory_error",
    "total_runtime_seconds",
    "mean_iterations_per_solve",
    "mean_relative_energy_error",
)

assert len(result.variants) == 16
assert len(result.solutions) == 16
assert len(summary_rows) == 16
assert len(metric_fields) == 5
assert tuple(row.key for row in summary_rows) == tuple(
    variant.key for variant in result.variants
)
assert all(
    len(result.solutions[variant.key]) == PARTICLE_COUNT
    for variant in result.variants
)
total_trajectory_count = sum(
    len(result.solutions[variant.key]) for variant in result.variants
)
assert total_trajectory_count == len(result.variants) * PARTICLE_COUNT

for variant in result.variants:
    for solution in result.solutions[variant.key]:
        assert solution.states.shape == (2, result.times.size)
        assert np.array_equal(solution.t, result.times)

metric_values = np.asarray(
    [[getattr(row, field) for field in metric_fields] for row in summary_rows],
    dtype=float,
)
assert metric_values.shape == (16, 5)
assert np.all(np.isfinite(metric_values))
assert np.all(metric_values >= 0.0)

print(f"Configurations: {len(result.variants)}")
print(f"Independent trajectories: {total_trajectory_count}")
print(f"Metric matrix shape: {metric_values.shape}")

## Comparison table: 16 rows by 5 metrics

The configuration label is the row identifier and is not part of the
$16\times5$ numerical matrix. The five metrics are:

1. mean trajectory RMS error against the reference over particles and time;
2. final-time particle RMS error;
3. total wall-clock integration time summed over the three independent runs;
4. mean nonlinear iterations per solve over particles, steps, and composed
   substeps; and
5. mean absolute relative generalized-energy error for $K=H+p_t$, using
   $p_t=\kappa$ in the shared-time extension and $p_t=k$ in the fully
   extended formulation.

Every error column is an error measure, so smaller values are better.

In [ ]:
display_records_table(
    summary_rows,
    columns=(
        ("label", "Configuration (row index)", None),
        ("mean_trajectory_error", "Mean trajectory RMS error", ".9e"),
        ("final_trajectory_error", "Final RMS error", ".9e"),
        ("total_runtime_seconds", "Total runtime [s]", ".6f"),
        ("mean_iterations_per_solve", "Mean iterations / solve", ".6f"),
        (
            "mean_relative_energy_error",
            "Mean relative generalized-energy error",
            ".9e",
        ),
    ),
)

## Synchronized 4 by 4 trajectory animation

Each panel corresponds to one configuration and contains the three trajectories
on the evolving effective PHI_2 background. The shared legend therefore has
exactly three stable initial-condition entries, one color per trajectory. Every background is the exact
field at the displayed saved time; the 101-frame view intentionally does not
resolve every fast carrier oscillation. Quick validation uses every available
saved sample, up to 21; with
the configured $[0,0.05]$ prefix this is six frames.

In [ ]:
animation_frame_count = (
    101 if FULL_STUDY else min(21, int(result.times.size))
)
animation = animate_abba4_configuration_trajectories(
    result,
    frames=animation_frame_count,
    interval=100,
    repeat=True,
)
display_animation(animation, embed_limit_mb=100.0)

## Interpretation notes

The full-study table is a controlled comparison on one potential, one set of
initial conditions, and one fixed integration grid. Runtime values are
descriptive single-pass measurements and should not be interpreted as a
machine-independent benchmark. Newton and Broyden iteration counts measure
nonlinear corrections, while their per-iteration costs can differ; the total
runtime column captures that distinction.

Generalized energy, rather than the time-dependent physical Hamiltonian alone,
is the conserved extended quantity. The quick profile is only an execution
and contract check; scientific conclusions must use `FULL_STUDY=True`.